# JAX Interpolation on Regular Grids

**Date:** 2026-04-02

## 概要

JAXで実装した補間ライブラリ（`jax_interp`）を使った1D・2D補間のデモ。
JIT対応・勾配計算可能で、scipy比で最大11倍高速。

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

jax.config.update('jax_enable_x64', True)

## 1D補間デモ

In [ ]:
# from jax_interp import interp1d, interp2d

# 均等グリッド (even=True がデフォルト)
xs = jnp.linspace(0, 2 * jnp.pi, 11)   # 端点2つでもOK: jnp.array([0., 2*pi])
values = jnp.sin(xs)
xq = jnp.linspace(0, 2 * jnp.pi, 200)

# JIT対応
f_jit = jax.jit(interp1d, static_argnames=['even', 'method', 'bc', 'oob'])

y_linear  = f_jit(xs, values, xq, method='linear')
y_cubic   = f_jit(xs, values, xq, method='cubic', bc='not-a-knot')
y_bspline = f_jit(xs, values, xq, method='bspline')

plt.figure(figsize=(10, 4))
plt.plot(xq, jnp.sin(xq), 'k--', label='true sin(x)')
plt.plot(xq, y_linear,  label='linear')
plt.plot(xq, y_cubic,   label='cubic/not-a-knot')
plt.plot(xq, y_bspline, label='bspline', linestyle=':')
plt.scatter(xs, values, color='red', zorder=5, label='knots')
plt.legend()
plt.title('1D Interpolation')
plt.show()

## 境界外処理 (oob)

In [ ]:
# グリッドを超えた範囲での挙動
xs_short = jnp.linspace(0, jnp.pi, 20)
vals = jnp.sin(xs_short)
xq_ext = jnp.linspace(-0.5, jnp.pi + 0.5, 300)

for oob in ['clamp', 'extrapolate', 'periodic', 'reflect']:
    y = interp1d(xs_short, vals, xq_ext, method='cubic', oob=oob)
    plt.plot(xq_ext, y, label=f'oob={oob}')

plt.axvline(0, color='gray', linestyle='--', alpha=0.5)
plt.axvline(jnp.pi, color='gray', linestyle='--', alpha=0.5)
plt.legend()
plt.title('Out-of-bounds behavior')
plt.show()

## 勾配計算

In [ ]:
# interp1d は values に対して微分可能
xs = jnp.linspace(0, 2 * jnp.pi, 11)
values = jnp.sin(xs)
xq_single = jnp.array([1.0])

grad_fn = jax.grad(lambda v: interp1d(xs, v, xq_single, method='linear').sum())
g = grad_fn(values)
print('gradient w.r.t. values:', g)

## 結果と考察

- cubic/not-a-knot・bspline はscipyと機械イプシロンレベルで一致
- JIT後は小〜大サイズで scipy より **1.3x〜11x** 高速
- `oob='reflect'` や `2d cubic` はscipyに相当なし（独自実装）
- 2D cubic（不均等グリッド）は誤差が大きく、要確認

## 参考資料

- [JAX documentation](https://jax.readthedocs.io/)
- [scipy.interpolate](https://docs.scipy.org/doc/scipy/reference/interpolate.html)